# Feature Extraction — Patch-Based + Cross-Modal Differential Features (CMDF)

**Pipeline Version: v3 — Patch × CMDF Fusion**

This notebook implements a combined strategy for 1p/19q co-deletion prediction:

| Stage | Method |
|-------|--------|
| Whole-tumour | T1Gd + T2 PyRadiomics (~214 features) → **Baseline** |
| Patch-based | 16×16×16 patch radiomics, aggregated stats + MIL → **Patch pipeline** |
| Cross-modal differential | Variance / L2 / Cosine / Outlier-deviation between T1Gd & T2 per patch → **CMDF fusion** |

**IMPORTANT SETUP:**
1. Open your shared Google Drive link → **Organize** → **Add shortcut** → **My Drive**
2. Name the shortcut exactly `UTSW`, so the dataset lives at
   `/content/drive/MyDrive/UTSW/UTSW-Glioma/<subject_id>/...` and the metadata TSV at
   `/content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv`
3. Run cells top-to-bottom

## 1. Install Requirements & Mount Drive

In [27]:
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git SimpleITK pandas tqdm pdfplumber scipy -q
from google.colab import drive
drive.mount('/content/drive')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import os
os.makedirs('/content/drive/MyDrive/Capstone_paper_001', exist_ok=True)
%cd /content/drive/MyDrive/Capstone_paper_001

/content/drive/MyDrive/Capstone_paper_001


## 2. config.py — Centralised Configuration

In [38]:
%%writefile config.py
"""
config.py – Centralized configuration for the 1p/19q co-deletion prediction pipeline.
Version 3: Patch-Based + Cross-Modal Differential Features (CMDF)
"""

import os

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  —  UTSW Glioma dataset (per-subject folders + TSV metadata)
# ──────────────────────────────────────────────────────────────────────────────
DATA_ROOT = "/content/drive/MyDrive/UTSW/UTSW-Glioma"          # per-subject imaging folders (e.g. BT0001)
METADATA_TSV_PATH = "/content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv"

IMAGES_ROOT_DIR = DATA_ROOT
PATIENT_RAW_DIR = DATA_ROOT     # kept for backward-compat with segmentation.py / patch_extraction.py

# Each sequence maps to a list of candidate subject-folder-relative filenames,
# tried in order (primary file first, ANTs-registered fallback second) — matches
# this dataset's on-disk naming convention.
IMAGE_PATH_TEMPLATES = {
    "t1Gd": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce_ants.nii.gz"),
    ],
    "t2": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2_ants.nii.gz"),
    ],
    # Not consumed by the CMDF pipeline (which only compares t1Gd vs t2), but
    # defined here in case SEQUENCES is extended later.
    "t1": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1_ants.nii.gz"),
    ],
    "flair": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_flair.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_fl_ants.nii.gz"),
    ],
}

# Segmentation mask candidates, tried in order, relative to each subject folder.
SEG_PATH_TEMPLATES = [
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "rtumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_FeTS.nii.gz"),
]

PROJECT_ROOT = "/content/drive/MyDrive/Capstone_paper_001"
OUTPUT_ROOT  = os.path.join(PROJECT_ROOT, "output")

PREPROCESS_DIR   = os.path.join(OUTPUT_ROOT, "preprocessed")
FEATURES_DIR     = os.path.join(OUTPUT_ROOT, "features")
AUGMENTED_DIR    = os.path.join(OUTPUT_ROOT, "augmented")
RESULTS_DIR      = os.path.join(OUTPUT_ROOT, "results")
MODELS_DIR       = os.path.join(OUTPUT_ROOT, "models")

# ──────────────────────────────────────────────────────────────────────────────
#  SEQUENCES
# ──────────────────────────────────────────────────────────────────────────────
# CMDF compares exactly these two modalities against each other. T1 / FLAIR are
# defined above in IMAGE_PATH_TEMPLATES but unused unless you extend the pipeline.
SEQUENCES = ["t1Gd", "t2"]

# ──────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ──────────────────────────────────────────────────────────────────────────────
TARGET_SPACING = (1.0, 1.0, 1.0)
N4_ITERATIONS  = [50, 50, 30, 20]
N4_CONVERGENCE_THRESHOLD = 1e-6
N4_SHRINK_FACTOR = 3

# ──────────────────────────────────────────────────────────────────────────────
#  PYRADIOMICS (shared by whole-tumour & patch extractors)
# ──────────────────────────────────────────────────────────────────────────────
PYRADIOMICS_PARAMS = {
    "binWidth": 25,
    "resampledPixelSpacing": None,
    "interpolator": "sitkBSpline",
    "normalizeScale": 1,
    "normalize": True,
    "removeOutliers": 3,
}

FEATURE_CLASSES = [
    "firstorder", "shape", "glcm", "glszm", "glrlm", "ngtdm", "gldm",
]

# ──────────────────────────────────────────────────────────────────────────────
#  VAE DATA AUGMENTATION
# ──────────────────────────────────────────────────────────────────────────────
VAE_LATENT_DIM    = 32
VAE_HIDDEN_UNITS  = [128, 64]
VAE_EPOCHS        = 200
VAE_BATCH_SIZE    = 16
VAE_LEARNING_RATE = 1e-3

# ──────────────────────────────────────────────────────────────────────────────
#  FEATURE SELECTION
# ──────────────────────────────────────────────────────────────────────────────
IG_TOP_K = 30

# ──────────────────────────────────────────────────────────────────────────────
#  1D-CNN
# ──────────────────────────────────────────────────────────────────────────────
CNN_FILTERS_1     = 64
CNN_FILTERS_2     = 128
CNN_KERNEL_SIZE   = 3
CNN_POOL_SIZE     = 2
CNN_DENSE_UNITS   = 128
CNN_DROPOUT       = 0.0
CNN_LEARNING_RATE = 0.0001
CNN_BATCH_SIZE    = 64
CNN_EPOCHS        = 50
CNN_NUM_CLASSES   = 2

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-VALIDATION
# ──────────────────────────────────────────────────────────────────────────────
N_FOLDS     = 5
RANDOM_SEED = 42

# ──────────────────────────────────────────────────────────────────────────────
#  LABEL / METADATA  (UTSW_Glioma_Metadata TSV, replaces the old labels PDF)
# ──────────────────────────────────────────────────────────────────────────────
COL_SUBJECT_ID = "Subject ID"
COL_LABEL      = "1p19Q CODEL"
PATIENT_ID_COL = "patient_id"
UNKNOWN_LABEL  = -1

LABEL_POSITIVE_VALUES = {
    "co-deleted", "codeleted", "codel", "1p/19q codeleted", "yes", "positive", "1",
}
LABEL_NEGATIVE_VALUES = {
    "non co-deleted", "non-codeleted", "not codeleted", "intact", "no", "negative", "0",
}

# ──────────────────────────────────────────────────────────────────────────────
#  PATCH-BASED
# ──────────────────────────────────────────────────────────────────────────────
PATCH_SIZE               = (16, 16, 16)
PATCH_MIN_TUMOR_FRACTION = 0.50
PATCH_STRIDE             = None
PATCH_FEATURES_DIR       = os.path.join(OUTPUT_ROOT, 'patch_features')
PATCH_RESULTS_DIR        = os.path.join(OUTPUT_ROOT, 'patch_results')
PATCH_FEATURE_CLASSES    = ['firstorder', 'glcm', 'glszm', 'glrlm', 'ngtdm', 'gldm']

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-MODAL DIFFERENTIAL FEATURES (CMDF)
# ──────────────────────────────────────────────────────────────────────────────
CMDF_FEATURES_DIR  = os.path.join(OUTPUT_ROOT, 'cmdf_features')
CMDF_RESULTS_DIR   = os.path.join(OUTPUT_ROOT, 'cmdf_results')

# Which differential metrics to compute between modalities
# Options: 'variance', 'l2', 'cosine', 'outlier_deviation'
CMDF_METRICS = ['variance', 'l2', 'cosine', 'outlier_deviation']

# ──────────────────────────────────────────────────────────────────────────────
#  ATTENTION MIL MODEL
# ──────────────────────────────────────────────────────────────────────────────
MIL_HIDDEN_DIM     = 128
MIL_ATTENTION_DIM  = 64
MIL_DROPOUT        = 0.3
MIL_LEARNING_RATE  = 1e-4
MIL_BATCH_SIZE     = 1
MIL_EPOCHS         = 80


Overwriting config.py


## 3. segmentation.py — (unchanged from baseline)

In [30]:
%%writefile segmentation.py
"""
segmentation.py – Tumour segmentation utility.
Updated to read the UTSW Glioma per-subject folder layout (SEG_PATH_TEMPLATES)
instead of the old separate SEG_MANUAL_DIR / SEG_AUTO_DIR + TCGA-* filename scheme.
"""

import os
import glob
import argparse
import numpy as np
import SimpleITK as sitk
from tqdm import tqdm

from config import (
    PATIENT_RAW_DIR, SEG_PATH_TEMPLATES,
)


def get_patient_ids():
    """Subject IDs = every subdirectory under PATIENT_RAW_DIR (e.g. BT0001, BT0002, ...)."""
    all_dirs = sorted(glob.glob(os.path.join(PATIENT_RAW_DIR, "*")))
    return [os.path.basename(d) for d in all_dirs if os.path.isdir(d)]


def get_mask_path(patient_id):
    """Try each SEG_PATH_TEMPLATES candidate in order, return the first that exists."""
    for template in SEG_PATH_TEMPLATES:
        candidate = template.format(subject_id=patient_id)
        if os.path.exists(candidate):
            return candidate
    return None


def load_mask(patient_id, binarize=True):
    path = get_mask_path(patient_id)
    if path is None:
        return None
    mask = sitk.ReadImage(path, sitk.sitkFloat32)
    if binarize:
        mask = sitk.BinaryThreshold(mask, lowerThreshold=0.5,
                                    upperThreshold=1e10,
                                    insideValue=1, outsideValue=0)
        mask = sitk.Cast(mask, sitk.sitkUInt8)
    return mask


def load_mask_as_numpy(patient_id, binarize=True):
    mask = load_mask(patient_id, binarize=binarize)
    return sitk.GetArrayFromImage(mask) if mask is not None else None


def verify_masks(max_patients=None):
    patient_ids = get_patient_ids()
    if max_patients:
        patient_ids = patient_ids[:max_patients]
    found = missing = empty = 0
    for pid in tqdm(patient_ids, desc="Verifying masks"):
        mask = load_mask(pid, binarize=True)
        if mask is None:
            missing += 1
        elif sitk.GetArrayFromImage(mask).sum() == 0:
            empty += 1
        else:
            found += 1
    total = found + missing + empty
    print(f"Masks: {found}/{total} OK, {missing} missing, {empty} empty.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--patients", type=int, default=None)
    args = parser.parse_args()
    verify_masks(max_patients=args.patients)


Overwriting segmentation.py


## 4. patch_extraction.py — Per-Patch Radiomics (baseline patch module)

In [31]:
%%writefile patch_extraction.py
"""
patch_extraction.py – 16×16×16 patch-based radiomics extraction.
Updated to read images via IMAGE_PATH_TEMPLATES (per-subject folder + filename
candidates with ANTs fallback) and labels from the UTSW Glioma metadata TSV
instead of the old PDF table.
Saves: output/patch_features/patch_radiomics_features.csv
"""

import os
import glob
import argparse
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm

import radiomics
from radiomics import featureextractor

from config import (
    PATIENT_RAW_DIR, IMAGE_PATH_TEMPLATES, SEQUENCES,
    PREPROCESS_DIR, PATCH_FEATURES_DIR,
    PYRADIOMICS_PARAMS, PATCH_FEATURE_CLASSES,
    PATCH_SIZE, PATCH_MIN_TUMOR_FRACTION, PATCH_STRIDE,
    METADATA_TSV_PATH, COL_SUBJECT_ID, COL_LABEL,
    LABEL_POSITIVE_VALUES, LABEL_NEGATIVE_VALUES,
)
from segmentation import get_mask_path

radiomics.setVerbosity(40)


def create_patch_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(**dict(PYRADIOMICS_PARAMS))
    extractor.disableAllFeatures()
    for fc in PATCH_FEATURE_CLASSES:
        extractor.enableFeatureClassByName(fc)
    return extractor


def compute_patch_origins(mask_arr, patch_size=PATCH_SIZE, stride=None,
                          min_tumor_fraction=PATCH_MIN_TUMOR_FRACTION):
    if stride is None:
        stride = patch_size
    pz, py, px = patch_size
    sz, sy, sx = stride
    D, H, W = mask_arr.shape
    nonzero = np.argwhere(mask_arr > 0)
    if len(nonzero) == 0:
        return []
    bbox_min = nonzero.min(axis=0)
    bbox_max = nonzero.max(axis=0) + 1
    origins = []
    for z in range(max(0, bbox_min[0] - pz), min(bbox_max[0], D - pz + 1), sz):
        for y in range(max(0, bbox_min[1] - py), min(bbox_max[1], H - py + 1), sy):
            for x in range(max(0, bbox_min[2] - px), min(bbox_max[2], W - px + 1), sx):
                patch_mask = mask_arr[z:z+pz, y:y+py, x:x+px]
                if patch_mask.sum() / patch_mask.size >= min_tumor_fraction:
                    origins.append((z, y, x))
    return origins


def _extract_single_patch(image_sitk, mask_sitk, origin, patch_size, extractor):
    z, y, x = origin
    pz, py, px = patch_size
    mask_arr = sitk.GetArrayFromImage(mask_sitk)
    pm = np.zeros_like(mask_arr, dtype=np.uint8)
    pm[z:z+pz, y:y+py, x:x+px] = mask_arr[z:z+pz, y:y+py, x:x+px]
    if pm.sum() == 0:
        return None
    pm_sitk = sitk.GetImageFromArray(pm)
    pm_sitk.CopyInformation(mask_sitk)
    try:
        result = extractor.execute(image_sitk, pm_sitk)
    except Exception:
        return None
    return {k: float(v) if hasattr(v, 'item') else v
            for k, v in result.items() if not k.startswith('diagnostics')}


def _find_image(patient_id, seq, use_preprocessed):
    if use_preprocessed:
        p = os.path.join(PREPROCESS_DIR, patient_id, f'{seq}_preprocessed.nii.gz')
        if os.path.exists(p):
            return p
    # Try each on-disk filename candidate for this sequence, in order
    # (primary file first, then the ANTs-registered fallback).
    for template in IMAGE_PATH_TEMPLATES.get(seq, []):
        candidate = template.format(subject_id=patient_id)
        if os.path.exists(candidate):
            return candidate
    return None


def extract_patient_patches(patient_id, extractor, use_preprocessed=True):
    """Extract per-patch features for both sequences. Returns list of dicts."""
    mask_path = get_mask_path(patient_id)
    if mask_path is None:
        return None
    mask_sitk = sitk.ReadImage(mask_path)
    mask_arr  = (sitk.GetArrayFromImage(mask_sitk) > 0).astype(np.uint8)
    if mask_arr.max() == 0:
        return None
    origins = compute_patch_origins(mask_arr)
    if not origins:
        return None
    bin_mask = sitk.GetImageFromArray(mask_arr)
    bin_mask.CopyInformation(mask_sitk)

    images = {}
    for seq in SEQUENCES:
        ip = _find_image(patient_id, seq, use_preprocessed)
        if ip is None or not os.path.exists(ip):
            return None
        images[seq] = sitk.ReadImage(ip, sitk.sitkFloat32)

    patch_rows = []
    for patch_idx, origin in enumerate(origins):
        row = {'patch_idx': patch_idx}
        for seq in SEQUENCES:
            feats = _extract_single_patch(images[seq], bin_mask, origin, PATCH_SIZE, extractor)
            if feats is None:
                row = None
                break
            for k, v in feats.items():
                row[f'{seq}_{k}'] = v
        if row is not None:
            patch_rows.append(row)
    return patch_rows or None


def _normalize_label(raw_value):
    """Map a raw metadata string to 0/1 using LABEL_POSITIVE_VALUES / LABEL_NEGATIVE_VALUES."""
    val = str(raw_value).strip().lower()
    if val in LABEL_POSITIVE_VALUES:
        return 1
    if val in LABEL_NEGATIVE_VALUES:
        return 0
    return None


def _load_labels():
    """Load subject_id -> 0/1 label from the UTSW Glioma metadata TSV."""
    label_map = {}
    try:
        meta = pd.read_csv(METADATA_TSV_PATH, sep='\t')
        for _, row in meta.iterrows():
            pid = str(row.get(COL_SUBJECT_ID, '')).strip()
            if not pid or pid.lower() == 'nan':
                continue
            label = _normalize_label(row.get(COL_LABEL, ''))
            if label is not None:
                label_map[pid] = label
    except Exception as e:
        print(f'Error reading metadata TSV: {e}')
    print(f'Loaded {len(label_map)} labels from {METADATA_TSV_PATH}.')
    return label_map


def get_eligible_subjects():
    """
    Cohort restriction: only subjects that (1) have a usable 1p19Q CODEL label
    in the metadata TSV AND (2) have a segmentation mask actually present on
    disk (checked via get_mask_path / SEG_PATH_TEMPLATES, not just a metadata
    flag, since that's the ground truth the extractor itself will use).
    Returns dict subject_id -> 0/1 label, restricted to that eligible set.
    """
    label_map = _load_labels()
    eligible = {pid: lbl for pid, lbl in label_map.items() if get_mask_path(pid) is not None}
    skipped_no_mask = len(label_map) - len(eligible)

    print(f'Metadata subjects with a valid 1p19Q CODEL label: {len(label_map)}')
    print(f'  -> of these, with a segmentation mask on disk : {len(eligible)}')
    print(f'  -> skipped (label present, no mask found)     : {skipped_no_mask}')
    return eligible


def extract_all_patches(max_patients=None, use_preprocessed=False):
    os.makedirs(PATCH_FEATURES_DIR, exist_ok=True)
    extractor = create_patch_extractor()
    label_map = get_eligible_subjects()
    patient_ids = list(label_map.keys())
    if max_patients:
        patient_ids = patient_ids[:max_patients]

    for pid in tqdm(patient_ids, desc="Extracting patch features"):
        patient_csv = os.path.join(PATCH_FEATURES_DIR, f"{pid}_patches.csv")

        # Checkpoint: Skip if already extracted
        if os.path.exists(patient_csv):
            continue

        patches = extract_patient_patches(pid, extractor, use_preprocessed=use_preprocessed)
        if not patches:
            continue

        for p in patches:
            p['patient_id'] = pid
            p['label'] = int(label_map[pid])

        # Save single patient immediately to Google Drive
        pd.DataFrame(patches).to_csv(patient_csv, index=False)

    # Aggregate all per-patient CSVs into master file
    all_files = glob.glob(os.path.join(PATCH_FEATURES_DIR, "*_patches.csv"))
    if not all_files:
        print("ERROR: No patch files found!")
        return pd.DataFrame()

    df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
    out = os.path.join(PATCH_FEATURES_DIR, "patch_radiomics_features.csv")
    df.to_csv(out, index=False)
    print(f"Saved aggregated patch features ({len(df)} total patches) -> {out}")
    return df


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--patients', type=int, default=None)
    parser.add_argument('--use_preprocessed', action='store_true')
    args = parser.parse_args()
    extract_all_patches(max_patients=args.patients,
                        use_preprocessed=args.use_preprocessed)


Overwriting patch_extraction.py


## 5. cmdf_features.py — Cross-Modal Differential Feature Computation *(NEW)*

This module computes per-patient **cross-modal disagreement signals** between T1Gd and T2 feature vectors, operating at both the whole-tumour level and per-patch level before statistical aggregation.

**Four differential metrics:**
| Metric | Formula | What it captures |
|---|---|---|
| Variance | `Var([t1_f, t2_f])` | Overall spread between modalities |
| L2 Distance | `‖t1_f − t2_f‖₂` | Magnitude of per-feature disagreement |
| Cosine Similarity | `cos(t1_f, t2_f)` | Directional alignment of feature vectors |
| Outlier Deviation | `|t1_f − t2_f| / (σ_t1 + σ_t2 + ε)` | Normalised extreme divergences |

In [32]:
%%writefile cmdf_features.py
"""
cmdf_features.py – Cross-Modal Differential Feature (CMDF) computation.

Computes disagreement signals between T1Gd and T2 radiomic features at the
patch level, then aggregates per patient.  The result is concatenated with
the standard patch aggregated features before classification.

Differential metrics
--------------------
  variance          : feature-wise variance across the two modalities
  l2                : per-feature absolute difference (L1/element-wise L2 proxy)
  cosine            : cosine similarity of the T1Gd vs T2 feature vector per patch
  outlier_deviation : (|t1−t2|) / (σ_dataset + ε) — highlights pathologically
                      large inter-modal discordances

Usage
-----
    from cmdf_features import compute_cmdf_from_patch_df
    cmdf_df = compute_cmdf_from_patch_df(patch_df)    # one row per patient
"""

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine as cosine_dist
from config import CMDF_METRICS, CMDF_FEATURES_DIR
import os


# ─────────────────────────────────────────────────────────────────────────────
#  LOW-LEVEL: DIFFERENTIAL METRICS FOR A SINGLE PATCH
# ─────────────────────────────────────────────────────────────────────────────

def _feature_variance(t1_vec: np.ndarray, t2_vec: np.ndarray) -> np.ndarray:
    """
    Feature-wise variance between T1Gd and T2 vectors.
    Returns a vector of length == n_features.
    """
    stacked = np.stack([t1_vec, t2_vec], axis=0)   # (2, F)
    return np.var(stacked, axis=0, ddof=0)           # (F,)


def _feature_l2(t1_vec: np.ndarray, t2_vec: np.ndarray) -> np.ndarray:
    """
    Element-wise absolute difference (per-feature L2 proxy).
    """
    return np.abs(t1_vec - t2_vec)                   # (F,)


def _cosine_similarity(t1_vec: np.ndarray, t2_vec: np.ndarray) -> float:
    """
    Cosine similarity (1 − cosine distance) between the full feature vectors.
    Returns a scalar in [-1, 1].
    """
    n1 = np.linalg.norm(t1_vec)
    n2 = np.linalg.norm(t2_vec)
    if n1 < 1e-10 or n2 < 1e-10:
        return 0.0
    return float(1.0 - cosine_dist(t1_vec, t2_vec))


def _outlier_deviation(t1_vec: np.ndarray, t2_vec: np.ndarray,
                       sigma_t1: np.ndarray, sigma_t2: np.ndarray,
                       eps: float = 1e-8) -> np.ndarray:
    """
    Normalised discordance: |t1 − t2| / (σ_t1 + σ_t2 + ε).
    Large values → the inter-modal difference is unusually large relative to
    each modality's own patch-to-patch variability.
    """
    return np.abs(t1_vec - t2_vec) / (sigma_t1 + sigma_t2 + eps)


# ─────────────────────────────────────────────────────────────────────────────
#  SPLIT PATCH-DF INTO T1Gd AND T2 FEATURE COLUMNS
# ─────────────────────────────────────────────────────────────────────────────

def _split_modality_columns(patch_df: pd.DataFrame):
    """
    Separate patch_df feature columns into t1Gd and t2 subsets.

    Returns
    -------
    t1_cols, t2_cols : lists of column names
    shared_names     : list of base feature names (without seq prefix)
    """
    meta_cols = {'patient_id', 'patch_idx', 'label'}
    all_feat  = [c for c in patch_df.columns if c not in meta_cols]

    t1_cols, t2_cols, shared = [], [], []
    for col in all_feat:
        if col.startswith('t1Gd_'):
            base = col[len('t1Gd_'):]
            t2_counterpart = f't2_{base}'
            if t2_counterpart in patch_df.columns:
                t1_cols.append(col)
                t2_cols.append(t2_counterpart)
                shared.append(base)

    return t1_cols, t2_cols, shared


# ─────────────────────────────────────────────────────────────────────────────
#  PATIENT-LEVEL CMDF AGGREGATION
# ─────────────────────────────────────────────────────────────────────────────

def _cmdf_for_patient(group: pd.DataFrame,
                      t1_cols, t2_cols, shared_names,
                      sigma_t1: np.ndarray, sigma_t2: np.ndarray,
                      active_metrics=None) -> dict:
    """
    Compute CMDF for all patches of one patient, then summarise with
    {mean, std, max} aggregation per metric.
    """
    if active_metrics is None:
        active_metrics = CMDF_METRICS

    T1 = group[t1_cols].values.astype(np.float32)  # (P, F)
    T2 = group[t2_cols].values.astype(np.float32)  # (P, F)
    P, F = T1.shape

    # Accumulate per-patch differential arrays
    patch_variance = np.zeros((P, F), dtype=np.float32)
    patch_l2       = np.zeros((P, F), dtype=np.float32)
    patch_cosine   = np.zeros(P,      dtype=np.float32)
    patch_outlier  = np.zeros((P, F), dtype=np.float32)

    for i in range(P):
        t1v = T1[i]
        t2v = T2[i]
        if 'variance' in active_metrics:
            patch_variance[i] = _feature_variance(t1v, t2v)
        if 'l2' in active_metrics:
            patch_l2[i] = _feature_l2(t1v, t2v)
        if 'cosine' in active_metrics:
            patch_cosine[i] = _cosine_similarity(t1v, t2v)
        if 'outlier_deviation' in active_metrics:
            patch_outlier[i] = _outlier_deviation(t1v, t2v, sigma_t1, sigma_t2)

    row = {}
    agg_fns = {'mean': np.mean, 'std': np.std, 'max': np.max}

    for stat_name, fn in agg_fns.items():
        if 'variance' in active_metrics:
            v = fn(patch_variance, axis=0)   # (F,)
            for j, base in enumerate(shared_names):
                row[f'cmdf_variance_{base}_{stat_name}'] = float(v[j])

        if 'l2' in active_metrics:
            v = fn(patch_l2, axis=0)
            for j, base in enumerate(shared_names):
                row[f'cmdf_l2_{base}_{stat_name}'] = float(v[j])

        if 'cosine' in active_metrics:
            row[f'cmdf_cosine_{stat_name}'] = float(fn(patch_cosine))

        if 'outlier_deviation' in active_metrics:
            v = fn(patch_outlier, axis=0)
            for j, base in enumerate(shared_names):
                row[f'cmdf_outlier_{base}_{stat_name}'] = float(v[j])

    return row


# ─────────────────────────────────────────────────────────────────────────────
#  PUBLIC API
# ─────────────────────────────────────────────────────────────────────────────

def compute_cmdf_from_patch_df(patch_df: pd.DataFrame,
                                active_metrics=None) -> pd.DataFrame:
    """
    Derive Cross-Modal Differential Features from the patch-level radiomics CSV.

    Parameters
    ----------
    patch_df       : DataFrame with columns patient_id, patch_idx, label,
                     t1Gd_<feat>, t2_<feat>  (one row per patch)
    active_metrics : list of metric names to compute (default: config.CMDF_METRICS)

    Returns
    -------
    cmdf_df : DataFrame with one row per patient,
              columns = patient_id, label, cmdf_<metric>_<feature>_<agg>
    """
    if active_metrics is None:
        active_metrics = CMDF_METRICS

    t1_cols, t2_cols, shared_names = _split_modality_columns(patch_df)
    print(f'  CMDF: {len(shared_names)} shared feature dimensions found.')
    print(f'  Active differential metrics: {active_metrics}')

    # Pre-compute dataset-level per-feature std for outlier_deviation
    T1_all = patch_df[t1_cols].values.astype(np.float32)
    T2_all = patch_df[t2_cols].values.astype(np.float32)
    sigma_t1 = T1_all.std(axis=0)   # (F,)
    sigma_t2 = T2_all.std(axis=0)

    rows = []
    for pid, group in patch_df.groupby('patient_id'):
        cmdf_row = _cmdf_for_patient(group, t1_cols, t2_cols, shared_names,
                                      sigma_t1, sigma_t2, active_metrics)
        cmdf_row['patient_id'] = pid
        cmdf_row['label'] = int(group['label'].iloc[0])
        rows.append(cmdf_row)

    cmdf_df = pd.DataFrame(rows)
    meta = ['patient_id', 'label']
    feat = [c for c in cmdf_df.columns if c not in meta]
    cmdf_df = cmdf_df[meta + feat]

    # Sanitize
    num_cols = cmdf_df.select_dtypes(include=[np.number]).columns
    cmdf_df[num_cols] = cmdf_df[num_cols].replace(
        [np.inf, -np.inf], np.nan).fillna(0.0)

    print(f'  CMDF shape: {cmdf_df.shape}  '
          f'({len(feat)} differential features per patient)')
    return cmdf_df


def fuse_patch_and_cmdf(patch_agg_df: pd.DataFrame,
                        cmdf_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge patient-level aggregated patch features with CMDF features.

    Parameters
    ----------
    patch_agg_df : output of mil_aggregation.aggregate_patient_features()
    cmdf_df      : output of compute_cmdf_from_patch_df()

    Returns
    -------
    fused_df : one row per patient, all patch-agg + CMDF columns combined
    """
    cmdf_feat = [c for c in cmdf_df.columns if c not in ('patient_id', 'label')]
    merged = patch_agg_df.merge(
        cmdf_df[['patient_id'] + cmdf_feat],
        on='patient_id', how='inner'
    )
    print(f'  Fused feature matrix: {merged.shape[0]} patients, '
          f'{len(merged.columns)} total columns '
          f'({len([c for c in merged.columns if c.startswith("cmdf_")])} CMDF columns)')
    return merged


def save_cmdf_csv(cmdf_df: pd.DataFrame, fused_df: pd.DataFrame):
    """Persist CMDF-only and fused feature CSVs."""
    os.makedirs(CMDF_FEATURES_DIR, exist_ok=True)
    c_path = os.path.join(CMDF_FEATURES_DIR, 'cmdf_only_features.csv')
    f_path = os.path.join(CMDF_FEATURES_DIR, 'fused_patch_cmdf_features.csv')
    cmdf_df.to_csv(c_path, index=False)
    fused_df.to_csv(f_path, index=False)
    print(f'  Saved CMDF-only  -> {c_path}')
    print(f'  Saved fused      -> {f_path}')

Overwriting cmdf_features.py


## 6. mil_aggregation.py — Patient-Level Aggregation (unchanged)

In [33]:
%%writefile mil_aggregation.py
"""
mil_aggregation.py – Patch-to-patient aggregation.
"""
import numpy as np
import pandas as pd


def aggregate_patient_features(patch_df, aggregation_stats=None):
    if aggregation_stats is None:
        aggregation_stats = ['mean', 'std', 'min', 'max', 'median']
    meta_cols = ['patient_id', 'patch_idx', 'label']
    feat_cols = [c for c in patch_df.columns if c not in meta_cols]
    rows = []
    for pid, group in patch_df.groupby('patient_id'):
        row = {'patient_id': pid,
               'label': int(group['label'].iloc[0]),
               'n_patches': len(group)}
        for stat in aggregation_stats:
            vals = getattr(group[feat_cols], stat)()
            if stat == 'std':
                vals = vals.fillna(0.0)
            for col in feat_cols:
                row[f'{col}_{stat}'] = vals[col]
        rows.append(row)
    agg_df = pd.DataFrame(rows)
    agg_meta = ['patient_id', 'label', 'n_patches']
    agg_feat = [c for c in agg_df.columns if c not in agg_meta]
    agg_df = agg_df[agg_meta + agg_feat]
    num = agg_df.select_dtypes(include=[np.number]).columns
    agg_df[num] = agg_df[num].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    print(f'  Aggregated {len(patch_df)} patches -> {len(agg_df)} patients, '
          f'{len(agg_feat)} features')
    return agg_df


def prepare_mil_bags(patch_df):
    meta_cols = ['patient_id', 'patch_idx', 'label']
    feat_cols = [c for c in patch_df.columns if c not in meta_cols]
    bags, labels, patient_ids = [], [], []
    for pid, group in patch_df.groupby('patient_id'):
        bags.append(group[feat_cols].values.astype(np.float32))
        labels.append(int(group['label'].iloc[0]))
        patient_ids.append(pid)
    return bags, np.array(labels), patient_ids

Overwriting mil_aggregation.py


## 7. Run Feature Extraction

Executes in two stages:
1. **Patch extraction** — extracts per-patch T1Gd + T2 radiomics (`patch_radiomics_features.csv`)
2. **CMDF computation** — derives cross-modal differential features and fuses with patch-aggregated features (`fused_patch_cmdf_features.csv`)

In [34]:
%%writefile run_feature_extraction_cmdf.py
"""
run_feature_extraction_cmdf.py – Orchestrates patch extraction + CMDF fusion.

Usage
-----
    python run_feature_extraction_cmdf.py                  # all patients
    python run_feature_extraction_cmdf.py --patients 5     # quick test
    python run_feature_extraction_cmdf.py --skip_patch     # reuse saved patch CSV
"""

import os
import argparse
import numpy as np
import pandas as pd

from config import PATCH_FEATURES_DIR, CMDF_FEATURES_DIR


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--patients',         type=int, default=None)
    parser.add_argument('--use_preprocessed', action='store_true')
    parser.add_argument('--skip_patch',       action='store_true',
                        help='Skip patch extraction if CSV already exists')
    args = parser.parse_args()

    patch_csv = os.path.join(PATCH_FEATURES_DIR, 'patch_radiomics_features.csv')

    # ── Stage 1: Patch extraction ──────────────────────────────────────────
    if args.skip_patch and os.path.exists(patch_csv):
        print(f'[SKIP] Patch CSV already exists: {patch_csv}')
        patch_df = pd.read_csv(patch_csv)
    else:
        print('\n' + '='*70)
        print('  STAGE 1: PATCH FEATURE EXTRACTION')
        print('='*70)
        from patch_extraction import extract_all_patches
        patch_df = extract_all_patches(
            max_patients=args.patients,
            use_preprocessed=args.use_preprocessed
        )

    if patch_df.empty:
        print('ERROR: No patch features. Aborting.')
        return

    # ── Stage 2: CMDF computation ──────────────────────────────────────────
    print('\n' + '='*70)
    print('  STAGE 2: CROSS-MODAL DIFFERENTIAL FEATURES (CMDF)')
    print('='*70)

    from cmdf_features import compute_cmdf_from_patch_df, fuse_patch_and_cmdf, save_cmdf_csv
    from mil_aggregation import aggregate_patient_features

    # 2a. Aggregate patches -> patient-level (standard stats)
    print('\n  [2a] Aggregating patch features to patient level...')
    agg_df = aggregate_patient_features(patch_df)

    # 2b. Compute cross-modal differentials
    print('\n  [2b] Computing CMDF...')
    cmdf_df = compute_cmdf_from_patch_df(patch_df)

    # 2c. Fuse
    print('\n  [2c] Fusing patch-aggregated + CMDF features...')
    fused_df = fuse_patch_and_cmdf(agg_df, cmdf_df)

    # 2d. Save
    save_cmdf_csv(cmdf_df, fused_df)

    print('\n✅ Feature extraction complete.')
    print(f'   Patients     : {len(fused_df)}')
    patch_feat = [c for c in fused_df.columns
                  if c not in ('patient_id', 'label', 'n_patches')
                  and not c.startswith('cmdf_')]
    cmdf_feat  = [c for c in fused_df.columns if c.startswith('cmdf_')]
    print(f'   Patch-agg features : {len(patch_feat)}')
    print(f'   CMDF features      : {len(cmdf_feat)}')
    print(f'   Total features     : {len(patch_feat) + len(cmdf_feat)}')


if __name__ == '__main__':
    main()

Overwriting run_feature_extraction_cmdf.py


In [35]:
# Run full extraction (patch + CMDF fusion)
!python run_feature_extraction_cmdf.py

# OR to reuse an already-extracted patch CSV:
# !python run_feature_extraction_cmdf.py --skip_patch


  STAGE 1: PATCH FEATURE EXTRACTION
Loaded 335 labels from /content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv.
Metadata subjects with a valid 1p19Q CODEL label: 335
  -> of these, with a segmentation mask on disk : 335
  -> skipped (label present, no mask found)     : 0
Extracting patch features:  88% 296/335 [00:03<00:00, 77.21it/s]
Traceback (most recent call last):
  File "/content/drive/MyDrive/Capstone_paper_001/run_feature_extraction_cmdf.py", line 82, in <module>
    main()
    ~~~~^^
  File "/content/drive/MyDrive/Capstone_paper_001/run_feature_extraction_cmdf.py", line 38, in main
    patch_df = extract_all_patches(
        max_patients=args.patients,
        use_preprocessed=args.use_preprocessed
    )
  File "/content/drive/MyDrive/Capstone_paper_001/patch_extraction.py", line 193, in extract_all_patches
    patches = extract_patient_patches(pid, extractor, use_preprocessed=use_preprocessed)
  File "/content/drive/MyDrive/Capstone_paper_001/patch_extraction.py", line 

## 8. Quick Sanity Check — Inspect Fused Feature Matrix

In [36]:
import pandas as pd
import numpy as np
import os
from config import CMDF_FEATURES_DIR

fused_path = os.path.join(CMDF_FEATURES_DIR, 'fused_patch_cmdf_features.csv')
fused_df   = pd.read_csv(fused_path)

patch_cols = [c for c in fused_df.columns
              if c not in ('patient_id', 'label', 'n_patches') and not c.startswith('cmdf_')]
cmdf_cols  = [c for c in fused_df.columns if c.startswith('cmdf_')]

print(f'Patients          : {len(fused_df)}')
print(f'Class distribution: {fused_df["label"].value_counts().to_dict()}')
print(f'Patch-agg features: {len(patch_cols)}')
print(f'CMDF features     : {len(cmdf_cols)}')
print(f'Total features    : {len(patch_cols) + len(cmdf_cols)}')
print()
print('Sample CMDF column names:')
for c in cmdf_cols[:12]:
    print(f'  {c}')

print('\nFused DataFrame head:')
fused_df[['patient_id', 'label', 'n_patches'] + patch_cols[:3] + cmdf_cols[:3]].head()

Patients          : 318
Class distribution: {0: 256, 1: 62}
Patch-agg features: 930
CMDF features     : 840
Total features    : 1770

Sample CMDF column names:
  cmdf_variance_original_firstorder_10Percentile_mean
  cmdf_variance_original_firstorder_90Percentile_mean
  cmdf_variance_original_firstorder_Energy_mean
  cmdf_variance_original_firstorder_Entropy_mean
  cmdf_variance_original_firstorder_InterquartileRange_mean
  cmdf_variance_original_firstorder_Kurtosis_mean
  cmdf_variance_original_firstorder_Maximum_mean
  cmdf_variance_original_firstorder_MeanAbsoluteDeviation_mean
  cmdf_variance_original_firstorder_Mean_mean
  cmdf_variance_original_firstorder_Median_mean
  cmdf_variance_original_firstorder_Minimum_mean
  cmdf_variance_original_firstorder_Range_mean

Fused DataFrame head:


,patient_id,label,n_patches,t1Gd_original_firstorder_10Percentile_mean,t1Gd_original_firstorder_90Percentile_mean,t1Gd_original_firstorder_Energy_mean,cmdf_variance_original_firstorder_10Percentile_mean,cmdf_variance_original_firstorder_90Percentile_mean,cmdf_variance_original_firstorder_Energy_mean
0,BT0001,0,31,1.800294,2.864535,19157.873054,0.212141,0.016537,30893808.0
1,BT0003,0,27,1.505132,2.081462,10437.910440,0.527714,0.222923,98854768.0
2,BT0005,0,41,2.007127,2.787326,21870.801444,0.087374,0.042195,13139217.0
3,BT0007,0,42,1.743097,2.453211,15030.365628,0.137832,0.087932,44282068.0
4,BT0011,1,42,1.998607,2.676653,17866.933026,0.059644,0.046341,17523232.0


In [24]:
def _find_image(patient_id, seq, use_preprocessed):
    if use_preprocessed:
        p = os.path.join(PREPROCESS_DIR, patient_id, f'{seq}_preprocessed.nii.gz')
        if os.path.exists(p):
            return p
    # Try each on-disk filename candidate for this sequence, in order
    # (primary file first, then the ANTs-registered fallback).
    for template in IMAGE_PATH_TEMPLATES.get(seq, []):
        candidate = template.format(subject_id=patient_id)
        if os.path.exists(candidate):
            return candidate
    return None


def extract_patient_patches(patient_id, extractor, use_preprocessed=True):
    """Extract per-patch features for both sequences. Returns list of dicts."""
    mask_path = get_mask_path(patient_id)
    if mask_path is None:
        return None
    mask_sitk = sitk.ReadImage(mask_path)
    mask_arr  = (sitk.GetArrayFromImage(mask_sitk) > 0).astype(np.uint8)
    if mask_arr.max() == 0:
        return None
    origins = compute_patch_origins(mask_arr)
    if not origins:
        return None
    bin_mask = sitk.GetImageFromArray(mask_arr)
    bin_mask.CopyInformation(mask_sitk)

    images = {}
    for seq in SEQUENCES:
        ip = _find_image(patient_id, seq, use_preprocessed)
        if ip is None or not os.path.exists(ip):
            return None
        images[seq] = sitk.ReadImage(ip, sitk.sitkFloat32)

    patch_rows = []
    for patch_idx, origin in enumerate(origins):
        row = {'patch_idx': patch_idx}
        for seq in SEQUENCES:
            feats = _extract_single_patch(images[seq], bin_mask, origin, PATCH_SIZE, extractor)
            if feats is None:
                row = None
                break
            for k, v in feats.items():
                row[f'{seq}_{k}'] = v
        if row is not None:
            patch_rows.append(row)
    return patch_rows or None


def _normalize_label(raw_value):
    """Map a raw metadata string to 0/1 using LABEL_POSITIVE_VALUES / LABEL_NEGATIVE_VALUES."""
    val = str(raw_value).strip().lower()
    if val in LABEL_POSITIVE_VALUES:
        return 1
    if val in LABEL_NEGATIVE_VALUES:
        return 0
    return None


def _load_labels():
    """Load subject_id -> 0/1 label from the UTSW Glioma metadata TSV."""
    label_map = {}
    try:
        meta = pd.read_csv(METADATA_TSV_PATH, sep='\t')
        for _, row in meta.iterrows():
            pid = str(row.get(COL_SUBJECT_ID, '')).strip()
            if not pid or pid.lower() == 'nan':
                continue
            label = _normalize_label(row.get(COL_LABEL, ''))
            if label is not None:
                label_map[pid] = label
    except Exception as e:
        print(f'Error reading metadata TSV: {e}')
    print(f'Loaded {len(label_map)} labels from {METADATA_TSV_PATH}.')
    return label_map


def get_eligible_subjects():
    """
    Cohort restriction: only subjects that (1) have a usable 1p19Q CODEL label
    in the metadata TSV AND (2) have a segmentation mask actually present on
    disk (checked via get_mask_path / SEG_PATH_TEMPLATES, not just a metadata
    flag, since that's the ground truth the extractor itself will use).
    Returns dict subject_id -> 0/1 label, restricted to that eligible set.
    """
    label_map = _load_labels()
    eligible = {pid: lbl for pid, lbl in label_map.items() if get_mask_path(pid) is not None}
    skipped_no_mask = len(label_map) - len(eligible)

    print(f'Metadata subjects with a valid 1p19Q CODEL label: {len(label_map)}')
    print(f'  -> of these, with a segmentation mask on disk : {len(eligible)}')
    print(f'  -> skipped (label present, no mask found)     : {skipped_no_mask}')
    return eligible


def extract_all_patches(max_patients=None, use_preprocessed=False):
    os.makedirs(PATCH_FEATURES_DIR, exist_ok=True)
    extractor = create_patch_extractor()
    label_map = get_eligible_subjects()
    patient_ids = list(label_map.keys())
    if max_patients:
        patient_ids = patient_ids[:max_patients]

    for pid in tqdm(patient_ids, desc="Extracting patch features"):
        patient_csv = os.path.join(PATCH_FEATURES_DIR, f"{pid}_patches.csv")

        # Checkpoint: Skip if already extracted
        if os.path.exists(patient_csv):
            continue

        patches = extract_patient_patches(pid, extractor, use_preprocessed=use_preprocessed)
        if not patches:
            continue

        for p in patches:
            p['patient_id'] = pid
            p['label'] = int(label_map[pid])

        # Save single patient immediately to Google Drive
        pd.DataFrame(patches).to_csv(patient_csv, index=False)

    # Aggregate all per-patient CSVs into master file
    all_files = glob.glob(os.path.join(PATCH_FEATURES_DIR, "*_patches.csv"))
    if not all_files:
        print("ERROR: No patch files found!")
        return pd.DataFrame()

    df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
    out = os.path.join(PATCH_FEATURES_DIR, "patch_radiomics_features.csv")
    df.to_csv(out, index=False)
    print(f"Saved aggregated patch features ({len(df)} total patches) -> {out}")
    return df


In [41]:

# ──────────────────────────────────────────────────────────────────────────────
#  PATHS  —  UTSW Glioma dataset (per-subject folders + TSV metadata)
# ──────────────────────────────────────────────────────────────────────────────
DATA_ROOT = "/content/drive/MyDrive/UTSW/UTSW-Glioma"          # per-subject imaging folders (e.g. BT0001)
METADATA_TSV_PATH = "/content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv"

IMAGES_ROOT_DIR = DATA_ROOT
PATIENT_RAW_DIR = DATA_ROOT     # kept for backward-compat with segmentation.py / patch_extraction.py

# Each sequence maps to a list of candidate subject-folder-relative filenames,
# tried in order (primary file first, ANTs-registered fallback second) — matches
# this dataset's on-disk naming convention.
IMAGE_PATH_TEMPLATES = {
    "t1Gd": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1ce_ants.nii.gz"),
    ],
    "t2": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t2_ants.nii.gz"),
    ],
    # Not consumed by the CMDF pipeline (which only compares t1Gd vs t2), but
    # defined here in case SEQUENCES is extended later.
    "t1": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_t1_ants.nii.gz"),
    ],
    "flair": [
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_flair.nii.gz"),
        os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "brain_fl_ants.nii.gz"),
    ],
}

# Segmentation mask candidates, tried in order, relative to each subject folder.
SEG_PATH_TEMPLATES = [
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "rtumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_manual_correction.nii.gz"),
    os.path.join(IMAGES_ROOT_DIR, "{subject_id}", "tumorseg_FeTS.nii.gz"),
]

PROJECT_ROOT = "/content/drive/MyDrive/Capstone_paper_001"
OUTPUT_ROOT  = os.path.join(PROJECT_ROOT, "output")

PREPROCESS_DIR   = os.path.join(OUTPUT_ROOT, "preprocessed")
FEATURES_DIR     = os.path.join(OUTPUT_ROOT, "features")
AUGMENTED_DIR    = os.path.join(OUTPUT_ROOT, "augmented")
RESULTS_DIR      = os.path.join(OUTPUT_ROOT, "results")
MODELS_DIR       = os.path.join(OUTPUT_ROOT, "models")

# ──────────────────────────────────────────────────────────────────────────────
#  SEQUENCES
# ──────────────────────────────────────────────────────────────────────────────
# CMDF compares exactly these two modalities against each other. T1 / FLAIR are
# defined above in IMAGE_PATH_TEMPLATES but unused unless you extend the pipeline.
SEQUENCES = ["t1Gd", "t2"]

# ──────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ──────────────────────────────────────────────────────────────────────────────
TARGET_SPACING = (1.0, 1.0, 1.0)
N4_ITERATIONS  = [50, 50, 30, 20]
N4_CONVERGENCE_THRESHOLD = 1e-6
N4_SHRINK_FACTOR = 3

# ──────────────────────────────────────────────────────────────────────────────
#  PYRADIOMICS (shared by whole-tumour & patch extractors)
# ──────────────────────────────────────────────────────────────────────────────
PYRADIOMICS_PARAMS = {
    "binWidth": 25,
    "resampledPixelSpacing": None,
    "interpolator": "sitkBSpline",
    "normalizeScale": 1,
    "normalize": True,
    "removeOutliers": 3,
}

FEATURE_CLASSES = [
    "firstorder", "shape", "glcm", "glszm", "glrlm", "ngtdm", "gldm",
]

# ──────────────────────────────────────────────────────────────────────────────
#  VAE DATA AUGMENTATION
# ──────────────────────────────────────────────────────────────────────────────
VAE_LATENT_DIM    = 32
VAE_HIDDEN_UNITS  = [128, 64]
VAE_EPOCHS        = 200
VAE_BATCH_SIZE    = 16
VAE_LEARNING_RATE = 1e-3

# ──────────────────────────────────────────────────────────────────────────────
#  FEATURE SELECTION
# ──────────────────────────────────────────────────────────────────────────────
IG_TOP_K = 30

# ──────────────────────────────────────────────────────────────────────────────
#  1D-CNN
# ──────────────────────────────────────────────────────────────────────────────
CNN_FILTERS_1     = 64
CNN_FILTERS_2     = 128
CNN_KERNEL_SIZE   = 3
CNN_POOL_SIZE     = 2
CNN_DENSE_UNITS   = 128
CNN_DROPOUT       = 0.0
CNN_LEARNING_RATE = 0.0001
CNN_BATCH_SIZE    = 64
CNN_EPOCHS        = 50
CNN_NUM_CLASSES   = 2

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-VALIDATION
# ──────────────────────────────────────────────────────────────────────────────
N_FOLDS     = 5
RANDOM_SEED = 42

# ──────────────────────────────────────────────────────────────────────────────
#  LABEL / METADATA  (UTSW_Glioma_Metadata TSV, replaces the old labels PDF)
# ──────────────────────────────────────────────────────────────────────────────
COL_SUBJECT_ID = "Subject ID"
COL_LABEL      = "1p19Q CODEL"
PATIENT_ID_COL = "patient_id"
UNKNOWN_LABEL  = -1

LABEL_POSITIVE_VALUES = {
    "co-deleted", "codeleted", "codel", "1p/19q codeleted", "yes", "positive", "1",
}
LABEL_NEGATIVE_VALUES = {
    "non co-deleted", "non-codeleted", "not codeleted", "intact", "no", "negative", "0",
}

# ──────────────────────────────────────────────────────────────────────────────
#  PATCH-BASED
# ──────────────────────────────────────────────────────────────────────────────
PATCH_SIZE               = (16, 16, 16)
PATCH_MIN_TUMOR_FRACTION = 0.50
PATCH_STRIDE             = None
PATCH_FEATURES_DIR       = os.path.join(OUTPUT_ROOT, 'patch_features')
PATCH_RESULTS_DIR        = os.path.join(OUTPUT_ROOT, 'patch_results')
PATCH_FEATURE_CLASSES    = ['firstorder', 'glcm', 'glszm', 'glrlm', 'ngtdm', 'gldm']

# ──────────────────────────────────────────────────────────────────────────────
#  CROSS-MODAL DIFFERENTIAL FEATURES (CMDF)
# ──────────────────────────────────────────────────────────────────────────────
CMDF_FEATURES_DIR  = os.path.join(OUTPUT_ROOT, 'cmdf_features')
CMDF_RESULTS_DIR   = os.path.join(OUTPUT_ROOT, 'cmdf_results')

# Which differential metrics to compute between modalities
# Options: 'variance', 'l2', 'cosine', 'outlier_deviation'
CMDF_METRICS = ['variance', 'l2', 'cosine', 'outlier_deviation']

# ──────────────────────────────────────────────────────────────────────────────
#  ATTENTION MIL MODEL
# ──────────────────────────────────────────────────────────────────────────────
MIL_HIDDEN_DIM     = 128
MIL_ATTENTION_DIM  = 64
MIL_DROPOUT        = 0.3
MIL_LEARNING_RATE  = 1e-4
MIL_BATCH_SIZE     = 1
MIL_EPOCHS         = 80

In [43]:

def get_patient_ids():
    """Subject IDs = every subdirectory under PATIENT_RAW_DIR (e.g. BT0001, BT0002, ...)."""
    all_dirs = sorted(glob.glob(os.path.join(PATIENT_RAW_DIR, "*")))
    return [os.path.basename(d) for d in all_dirs if os.path.isdir(d)]


def get_mask_path(patient_id):
    """Try each SEG_PATH_TEMPLATES candidate in order, return the first that exists."""
    for template in SEG_PATH_TEMPLATES:
        candidate = template.format(subject_id=patient_id)
        if os.path.exists(candidate):
            return candidate
    return None


def load_mask(patient_id, binarize=True):
    path = get_mask_path(patient_id)
    if path is None:
        return None
    mask = sitk.ReadImage(path, sitk.sitkFloat32)
    if binarize:
        mask = sitk.BinaryThreshold(mask, lowerThreshold=0.5,
                                    upperThreshold=1e10,
                                    insideValue=1, outsideValue=0)
        mask = sitk.Cast(mask, sitk.sitkUInt8)
    return mask


def load_mask_as_numpy(patient_id, binarize=True):
    mask = load_mask(patient_id, binarize=binarize)
    return sitk.GetArrayFromImage(mask) if mask is not None else None


def verify_masks(max_patients=None):
    patient_ids = get_patient_ids()
    if max_patients:
        patient_ids = patient_ids[:max_patients]
    found = missing = empty = 0
    for pid in tqdm(patient_ids, desc="Verifying masks"):
        mask = load_mask(pid, binarize=True)
        if mask is None:
            missing += 1
        elif sitk.GetArrayFromImage(mask).sum() == 0:
            empty += 1
        else:
            found += 1
    total = found + missing + empty
    print(f"Masks: {found}/{total} OK, {missing} missing, {empty} empty.")

In [45]:
import os
import glob
import pandas as pd
from tqdm import tqdm
from radiomics import featureextractor

# ==========================================
# 1. Configuration & Output Paths
# ==========================================
OUTPUT_DIR = "/content/drive/MyDrive/Capstone_paper_001/output/features"
TEMP_DIR = os.path.join(OUTPUT_DIR, "temp_baseline_radiomics")
FINAL_CSV = os.path.join(OUTPUT_DIR, "radiomics_features.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

# ==========================================
# 2. PyRadiomics Extractor Setup
# ==========================================
# ==========================================
# 2. PyRadiomics Extractor Setup
# ==========================================
# Define settings directly
settings = {
    'binWidth': 25,
    'resampledPixelSpacing': None,  # e.g., [1, 1, 1] if spatial normalization is needed
    'interpolator': 'sitkBSpline'
}

# Pass them as keyword arguments using **
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)

# Enable feature classes
extractor.enableAllImageTypes()   # Enables Original, Wavelet, Log, etc.
extractor.enableAllFeatures()     # Firstorder, Shape, GLCM, RLM, GLSZM, etc.

# ==========================================
# 3. Patient Path Helper (Adjust to match your folder structure)
# ==========================================
# ==========================================
# 3. Patient Path Helper (Robust Version)
# ==========================================
def get_patient_files(pid):
    """
    Dynamically searches for mask and modality files in the patient folder.
    """
    # Make sure this matches your actual data directory path!
    base_dir = os.path.join(DATA_DIR, str(pid))

    # Auto-detect the mask file (looking for 'seg' or 'mask' in the filename)
    mask_files = glob.glob(os.path.join(base_dir, "*seg*.nii*")) + \
                 glob.glob(os.path.join(base_dir, "*mask*.nii*"))

    # Auto-detect T1Gd (often named 't1ce' or 't1gd') and T2 files
    t1gd_files = glob.glob(os.path.join(base_dir, "*t1gd*.nii*")) + \
                 glob.glob(os.path.join(base_dir, "*t1ce*.nii*"))
    t2_files = glob.glob(os.path.join(base_dir, "*t2*.nii*"))

    mask_path = mask_files[0] if mask_files else ""

    image_paths = {}
    if t1gd_files:
        image_paths['t1gd'] = t1gd_files[0]
    if t2_files:
        image_paths['t2'] = t2_files[0]

    return image_paths, mask_path

# ==========================================
# 4. Feature Extraction Function per Subject
# ==========================================
def extract_baseline_features_for_patient(pid, label_val=None):
    image_paths, mask_path = get_patient_files(pid)

    if not os.path.exists(mask_path):
        print(f"Warning: Mask missing for patient {pid}, skipping.")
        return None

    patient_row = {'patient_id': pid}
    if label_val is not None:
        patient_row['label'] = label_val

    for modality, img_path in image_paths.items():
        if not os.path.exists(img_path):
            continue

        try:
            # Extract features for the entire ROI/mask
            result = extractor.execute(img_path, mask_path)

            # Keep radiomics features, skip PyRadiomics metadata/diagnostics
            for key, val in result.items():
                if not key.startswith("diagnostics_"):
                    patient_row[f"{modality}_{key}"] = float(val)
        except Exception as e:
            print(f"Error processing {pid} on modality {modality}: {e}")

    return patient_row

# ==========================================
# 5. Main Loop with Checkpointing
# ==========================================
def run_baseline_extraction(patient_list, label_map=None):
    print(f"Starting baseline radiomics extraction for {len(patient_list)} subjects...")

    for pid in tqdm(patient_list, desc="Extracting Baseline Radiomics"):
        patient_csv = os.path.join(TEMP_DIR, f"{pid}_baseline.csv")

        # Checkpoint: Skip if already extracted in a previous session
        if os.path.exists(patient_csv):
            continue

        label_val = label_map.get(pid, None) if label_map else None
        features = extract_baseline_features_for_patient(pid, label_val)

        if features and len(features) > 2:  # Ensure features were extracted
            pd.DataFrame([features]).to_csv(patient_csv, index=False)

    # Combine per-patient CSVs into the final baseline dataset
    temp_files = glob.glob(os.path.join(TEMP_DIR, "*_baseline.csv"))
    if not temp_files:
        print("No feature files were found or generated.")
        return None

    df_all = pd.concat([pd.read_csv(f) for f in temp_files], ignore_index=True)
    df_all.to_csv(FINAL_CSV, index=False)

    print(f"\nSuccess! Saved baseline radiomics features ({len(df_all)} patients, {df_all.shape[1]} features) -> {FINAL_CSV}")
    return df_all

# ==========================================
# 6. Execution Call
# ==========================================
# Retrieve your list of patient IDs and labels (using your existing pipeline logic)
label_map = get_eligible_subjects()  # Re-uses your existing helper function
patient_ids = list(label_map.keys())

# Run extraction
baseline_df = run_baseline_extraction(patient_ids, label_map=label_map)

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline'}


Loaded 335 labels from /content/drive/MyDrive/UTSW/UTSW_Glioma_Metadata-2-1.tsv.
Metadata subjects with a valid 1p19Q CODEL label: 335
  -> of these, with a segmentation mask on disk : 335
  -> skipped (label present, no mask found)     : 0
Starting baseline radiomics extraction for 335 subjects...


Extracting Baseline Radiomics:   0%|          | 0/335 [00:00<?, ?it/s]


NameError: name 'DATA_DIR' is not defined